# 1. Bidirection

## Weird Side-Effect

let's take a look over the final skipping patterns by overlaing them over the attention matrix $P = softmax(QK^T)$

<img src="undirectional_examples/t_0156.png" width="250" height="250" />
<img src="undirectional_examples/t_0156 2.png" width="250" height="250" />
<img src="undirectional_examples/t_0156 3.png" width="250" height="250" />
<img src="undirectional_examples/t_0156 4.png" width="250" height="250" />
<img src="undirectional_examples/t_0156 5.png" width="250" height="250" />
<img src="undirectional_examples/t_0156 6.png" width="250" height="250" />

- *the white squares represents tiles we didn't skip*
- *the more bright the higher the value in the $P$ matrix*

for some reason we get almost no skips right to the diagonal...

## Why?

this artifact is a result of the order we process the tiles.\
how does it work? lets zoom over:

<img src="undirectional_examples/unidirection.drawio4.svg" width="1000" height="400" />

internally we process each row independently and iterate over the tiles from right to left.\
as a result we only find the max value after passing the diagonal.\
meaning, only after that it is possible for us to get any skips!

in other words,\
**the start of the range we don't skip is always bigger then it should while the end of the range is correct.**

## Possible Solutions (optional, not sure if it's needed)

currently, we can roughly describe the algorithm like this:

In [ ]:
def lite_attention_psudo_code(QK_row, threshold):
    prev_max = -float('inf')
    for element in reversed(QK_row):
        skip = element - prev_max <= threshold
        prev_max = max(prev_max, element)

and ideally, if we would have the max value for each row ahead of time we would get the most amount of skips possible.

In [ ]:
def ideal_lite_attention(QK_row, threshold, max_value):
    for element in QK_row:
        skip = element - max_value <= threshold

1. Radial K-ordering - iterating from the diagonal, one to the right and one to the left. issue: heuristic (give an example for attention head where it's worse).
2. Max Location Save - saving the location of the tile with the max value and always start the iteration from it. issue: there is 128~ rows in each tile. should we save the max of each? if not it's another heuristic.

## Solution - Bidirectional Iteration

if we switch the iteration order between the time steps we would correct the "start range bias".\
[show an example over 3 consecutive timesteps]

## Implementation - Producer Consumer

while prototyping and exploring the different solutions it became tricky and error prone to change the iteration order\
in the Producer and Consumer side.\
to solve this problem we modified the way the consumer and producer agreed on the iteration order and switched to a full producer consumer pattern.

previously the producer and consumer had an independent loop which determiners the iteration order.
[show psudo code with producer iter loop and consumer iter loop]
but we want the producer to be able to tell the consumer what tile to run.
[show the new approch. sending the tile via shared memory]

Producer Implicit Padding

# 2. LiteAttention - Low Level Optimization Tricks

in this blog we would mention a bunch of cool twiks and optimizations we added to LiteAttention.

## Succsessfull Optimizations

### Faster -INF padding

in LiteAttention (and Flash Attention) when the tile size dosn't devide perfecly the size of the $QK^T$ matrix\
we need to padd the extra values and set them to be `-INFINITY`.\
in addition, becuase we switched to full producer consumer pattern, we potantially need to padd every tile since we can't assume anything about\
the order the consumers get's it's tiles.

```c++
                #pragma unroll
                for (int n = 0; n < size<1>(tSrS_rowcol); ++n) {
                    if (int(get<Col>(t0ScS_rowcol(_0{}, n))) >= seqlenk_col_limit) {
                        #pragma unroll
                        for (int m = 0; m < size<0>(tSrS_rowcol); ++m) { tSrS_rowcol(m, n) = -INFINITY; }
                    }
                }
```
*[the code snippet is from flash attention 3](https://github.com/Dao-AILab/flash-attention/blob/120b30694d57792e0d58a33fbc05024c2c003714/hopper/mask.h#L69-L75)*

this `if` statment being translated to a SASS instruction called `FSEL` (floating point select):

<img src="blog_SASS_images/Screenshot 2026-03-05 at 16.52.29.png" width=500, height=144/>

which is equivalent to:

```python
R13 = R24 if P4 == True else -INF
```

we can also notice that the stall counter the compiler decided for these insturctions is 2 cycles.\
meaning this instruction throughput is 64 results per cycle.\
in addition, according to the [instruction throughput table](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html#throughput-of-native-arithmetic-instructions) provided by nvidia\
the throughput for floating point fused multiply and add (FFMA) is 128.

what we can learn from this is:\
**if we were able to express the line `x = y if p==true else -INFINITY` using FFMA operation, we could 2X the throughput for the masking operation.**

here is how we could do it:
```c++
    ElementType col_value = (int(get<Col>(t0ScS_rowcol(_0{}, n))) >= seqlenk_col_limit) * 2.0f; // 2 if p==true else 0
    #pragma unroll
    // most_negative_value * col_value overflow to -INFINITY when col_value == 2.
    for (int m = 0; m < size<0>(tSrS_rowcol); ++m) { tSrS_rowcol(m, n) += most_negative_value * col_value; }
```
the idea is to cause a controlled overflow which always result in -INFINITY when the padding condition is true.

now, looking at the resulting SASS:
<!-- <img src="blog_SASS_images/Screenshot 2026-03-05 at 15.55.40.png" width="500" height="250" /> -->
<img src="blog_SASS_images/Screenshot 2026-03-05 at 15.55.40.png" />

and as we can see, the stall for each of the padding instructions reduced to 1 cycle.

### Int To Float - Conversion Using Single Addition

in the int8 version of LiteAttention we calculate $QK$ with $Q$ and $K$ represented in `int8`
and the output is in `int32`. \
but how many of the bits are really used in each $s \in QK$?

$$
s = \sum^{\text{ head dim }}_{i = 0} q_i k_i
$$
because $q_i$ and $k_i$ each represented in `int8` the multiplication of them should consume at most 16 bits.\
as a result we can say that:
$$
(-128) * 127 < q_i k_i <= (-128)^2 \Rightarrow \sum^{\text{ head dim }}_{i = 0} (-128) 127 < s <= \sum^{\text{ head dim }}_{i = 0} 128^2
\Rightarrow
\\
(\text{head dim}) * (-128) 127 <= s <= (\text{head dim}) * 128^2
$$

meaning the max number of bits we need to represent each $s \in QK$ is $\log_2((\text{head dim}) * 128^2)$.\
in the H100 the biggest tensor core instruction instruction possible forces us to sum at most $256$ elements.\
effectivly that mean that we can replace the $\text{head dim}$ variable with $256$.
so finally we get that:
$$
\log_2((\text{head dim}) * 128^2) <= \log_2(256 * 128^2) = 22
$$
the bottom line is that we only need 22 bits for representing $s$

there is a very cool [trick for converting an int in the range $[0, 2^{23}]$ to a float](https://purplesyringa.moe/blog/fast-limited-range-conversion-between-ints-and-floats/)\
and we can extend this method to work on the ranges $[-2^{22}, 2^{22} - 1]$ by changing the `magic` number from this:
```c++
float magic = float(1 << 23)
```
to this:
```c++
float magic = float((1 << 23) + (1 << 22))
```

now we have a new way to convert $s$ from `int` to `float` by simply doing:
```c++
float int2float(int s){
    constexpr float magic_float = float((1 << 23) + (1 << 22));
    constexpr int magic_int = reinterpret_bits<int>(magic_float);
    float s_float = reinterpret_bits<float>(s + magic_int) - magic_float;
    return s_float;
}
```

this method uses one integer addition and one floating point addition.\
is it possible to lower it into one instruction?\
the answer is yes, but only because we can merge the conversion with another operation that happens right after we convert to `float`.\


it LiteAttention we actually run something more similar to this:
```c++
// this operation we do once for every 44~ elements
float log2e = log2(e);
float max_scaled = log2e * max_value;

// for each element s
float s_float = int2float(s);
// this uses only one SASS instruction called FFMA (fused multiply add)
float p = s_float * log2e - max_scaled
```


we can fuse the `- magic_float` operation with `max_scaled` and get:
```c++
// this operation we do once for every 44~ elements

constexpr float magic_float = float((1 << 23) + (1 << 22));
constexpr int magic_int = reinterpret_bits<int>(magic_float);

float log2e = log2(e);
float max_scaled = log2e * max_value + magic_float;

// for each element s
float s_float = int2float(s);
float s_almost_float = reinterpret_bits<float>(s + magic_int)
// this uses only one SASS instruction called FFMA (fused multiply add)
float p = s_float * log2e - max_scaled
```

after this int2float conversion costs us one int addition while previouslly we used the built casting instruciton.\
comparing each instruction throughput by examining the [througput table](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html#arithmetic-instructions-throughput) we get that:\
`(float) s` throughput is 64 results per cycle and for `s + magic_int` it's 128 which 2X.

### Instruction Mixing & Dependency Breaks

### Weird `warpgroup_wait` Optimization?
`shfl` between syncs??

## Failed Optimizations (but too cool to not mention)

### Swizzle During INT8 Quantization [optional, needs to verify further]

### Max Reduction & Exp2 Dependency Break!
using int max reduction while doing exp2 over the fractional parts.

### Int2Float Full Dependency Break!
setup the fragment with the magic constant. failed because the compiler feel very smart...